# AlertaBairro: especialização do modelo com LoRA

**Disciplina:** IA Generativa, Global Solution FIAP 2026.2 (CitizenTech)

**Objetivo.** Ajustar o `Qwen2.5-3B-Instruct`, o mesmo modelo que o site usa no Ollama, para seguir o **contrato do assistente**:
- responder só com os trechos recebidos, com o número exato;
- copiar em `fontes` o id do trecho;
- recusar quando os trechos não respondem;
- devolver sempre JSON válido.

**Por que LoRA.** Na avaliação da RAG simples (`avaliacao/resultados/placar_rag.jsonl`), o modelo base entendeu o conteúdo, mas errou o formato:
- inventou ids como `sub:mes2026-05` e `doc:limites_dados#5`;
- 14% das respostas foram barradas pela validação.

O LoRA treina só adaptadores pequenos, então cabe numa GPU T4 gratuita do Colab, e o resultado pode ser exportado em GGUF para rodar no mesmo Ollama.

**Dados.** `treino/dados_treino.jsonl` e `treino/dados_validacao.jsonl` são gerados por `treino/gerar_dados_treino.py`:
- usam o mesmo prompt e a mesma busca BM25 do site;
- não contêm nenhuma das 50 perguntas de teste (o script mede a similaridade e descarta as parecidas).

**Como rodar.** Menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois, rode as células em ordem.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Instalar o Unsloth
Biblioteca de ajuste fino eficiente (LoRA com o modelo em 4 bits). Leva alguns minutos.

In [ ]:
!pip install -q unsloth

## 3. Baixar o repositório (dados de treino e código de validação do site)

In [ ]:
!rm -rf AlertaBairro
!git clone -q -b genai-avaliacao --depth 1 https://github.com/GeovannaSCunha/AlertaBairro.git
!wc -l AlertaBairro/treino/dados_treino.jsonl AlertaBairro/treino/dados_validacao.jsonl
!cat AlertaBairro/treino/estatisticas.json | head -20

## 4. Carregar o modelo base em 4 bits

In [ ]:
import sys, json, time, torch
sys.path.insert(0, "AlertaBairro/backend")
from unsloth import FastLanguageModel

SEMENTE = 42
MAX_TOKENS = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_TOKENS,
    load_in_4bit=True,
)
print(model.config._name_or_path, "| GPU:", torch.cuda.get_device_name(0))

## 5. Função de avaliação no conjunto de validação

Mede, nos exemplos de validação (que não entram no treino):
- **JSON válido:** usa o mesmo validador do site, `app.assistente.llm.validar_saida`, que barra JSON quebrado e ids fora do contexto;
- **fontes exatas:** o modelo citou exatamente os ids esperados;
- **recusa correta:** o campo `recusou` bate com o esperado.

In [ ]:
from app.assistente.llm import validar_saida, ErroLLM

def carregar(caminho):
    return [json.loads(l) for l in open(caminho, encoding="utf-8") if l.strip()]

validacao = carregar("AlertaBairro/treino/dados_validacao.jsonl")

def ids_do_contexto(texto_usuario):
    return {l[1:l.index("]")] for l in texto_usuario.splitlines() if l.startswith("[")}

def avaliar(model, exemplos, rotulo):
    FastLanguageModel.for_inference(model)
    json_ok = fontes_ok = recusa_ok = 0
    amostras = []
    for ex in exemplos:
        msgs = ex["messages"][:-1]
        esperado = json.loads(ex["messages"][-1]["content"])
        entrada = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            saida = model.generate(input_ids=entrada, max_new_tokens=300, do_sample=False)
        texto = tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True)
        try:
            obj = validar_saida(texto, ids_do_contexto(msgs[1]["content"]))
            json_ok += 1
            fontes_ok += sorted(obj["fontes"]) == sorted(esperado["fontes"])
            recusa_ok += obj["recusou"] == esperado["recusou"]
        except ErroLLM as exc:
            texto = f"[barrado: {exc}] {texto}"
        amostras.append(texto)
    n = len(exemplos)
    print(f"{rotulo}: JSON válido {json_ok}/{n} ({json_ok/n:.0%}) | fontes exatas {fontes_ok}/{n} ({fontes_ok/n:.0%}) "
          f"| recusa correta {recusa_ok}/{n} ({recusa_ok/n:.0%})")
    for t in amostras[:3]:
        print("  ->", t[:200])
    return {"json_valido": json_ok / n, "fontes_exatas": fontes_ok / n, "recusa_correta": recusa_ok / n}

## 6. Avaliar o modelo base, antes do treino

In [ ]:
antes = avaliar(model, validacao, "ANTES (modelo base)")

## 7. Adicionar os adaptadores LoRA
Só cerca de 1% dos parâmetros são treinados. O modelo base fica congelado.

In [ ]:
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEMENTE,
)
treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {treinaveis:,} de {total:,} ({treinaveis/total:.2%})")

## 8. Preparar os dados no formato de conversa do Qwen

In [ ]:
from datasets import Dataset

def para_texto(exemplos):
    return [{"text": tokenizer.apply_chat_template(e["messages"], tokenize=False)} for e in exemplos]

treino = carregar("AlertaBairro/treino/dados_treino.jsonl")
ds_treino = Dataset.from_list(para_texto(treino))
ds_validacao = Dataset.from_list(para_texto(validacao))
tamanhos = [len(tokenizer(t["text"]).input_ids) for t in ds_treino]
print(f"treino: {len(ds_treino)} exemplos | validação: {len(ds_validacao)} | tokens por exemplo: média {sum(tamanhos)/len(tamanhos):.0f}, máx {max(tamanhos)}")

## 9. Treinar

A perda é calculada **só na resposta do assistente**, com `train_on_responses_only`: o modelo aprende a responder, e não a repetir o prompt.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_treino,
    eval_dataset=ds_validacao,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_TOKENS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="epoch",
        output_dir="saida_lora",
        seed=SEMENTE,
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
inicio = time.time()
estatisticas = trainer.train()
print(f"Tempo de treino: {(time.time()-inicio)/60:.1f} min | perda final de treino: {estatisticas.training_loss:.4f}")
for registro in trainer.state.log_history:
    if "eval_loss" in registro:
        print(f"época {registro['epoch']:.0f}: perda na validação = {registro['eval_loss']:.4f}")

## 10. Avaliar o modelo ajustado, depois do treino

In [ ]:
depois = avaliar(model, validacao, "DEPOIS (LoRA)")
print()
print(f"{'métrica':<16}{'antes':>8}{'depois':>8}")
for k in antes:
    print(f"{k:<16}{antes[k]:>8.0%}{depois[k]:>8.0%}")

## 11. Exportar para o Ollama (GGUF q4_k_m) e salvar no Google Drive

O modelo é exportado com os adaptadores já fundidos, na mesma quantização (`q4_k_m`) do `qwen2.5:3b-instruct` que roda no Ollama. Assim a comparação é justa. O adaptador LoRA sozinho (poucos MB) também é salvo.

In [ ]:
model.save_pretrained("alertabairro_lora_adaptador")
tokenizer.save_pretrained("alertabairro_lora_adaptador")
model.save_pretrained_gguf("alertabairro_lora_gguf", tokenizer, quantization_method="q4_k_m")
!ls -la alertabairro_lora_adaptador alertabairro_lora_gguf

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/AlertaBairro_LoRA
!cp -v alertabairro_lora_gguf/*.gguf /content/drive/MyDrive/AlertaBairro_LoRA/
!cp -rv alertabairro_lora_adaptador /content/drive/MyDrive/AlertaBairro_LoRA/
json.dump({"antes": antes, "depois": depois}, open("/content/drive/MyDrive/AlertaBairro_LoRA/metricas_validacao.json", "w"), indent=2)
!ls -la /content/drive/MyDrive/AlertaBairro_LoRA